### Allscripts Sunrise (SCM) - Observation Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur (observation records)
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur (links observations to documents, has RecordedDtm)
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur (links documents to patients via ClientGUID)

**Mapping Table:**
- _exponent.results_store.omop_mapping_scm_observation_row_results_v1 (observation concept mappings via ObsItemGUID)

**Join Path:**
1. cv3observationcur → cv3observationdocumentcur (via obs.GUID = obsdoc.ObservationGUID)
2. cv3observationdocumentcur → cv3clientdocumentcur (via obsdoc.OwnerGUID = doc.GUID)
3. cv3clientdocumentcur → source_to_person (via doc.ClientGUID)

**Strategy:**
- Use cv3observationcur for observation data (non-numeric values)
- Use RecordedDtm from cv3observationdocumentcur for observation date
- Map ObsItemGUID to OMOP observation concepts via mapping table
- Use UnitOfMeasure for unit information
- observation_type_concept_id = 32817 (EHR)

**Note:**
- Sunrise stores observations and measurements in cv3observationcur
- This notebook captures non-numeric observation types
- See allscripts_scm_measurement.ipynb for numeric values

In [0]:
source = 'allscripts_scm'

# Transformation

In [ ]:
%sql
-- Create silver_observation temp view
-- Source: allscripts_scm
-- Join path: cv3observationcur → cv3observationdocumentcur → cv3clientdocumentcur → person

CREATE OR REPLACE TEMP VIEW silver_observation AS
SELECT
  stp.person_id,
  COALESCE(obs_concept.concept_id, 0) AS observation_concept_id,
  CAST(obsdoc.RecordedDtm AS DATE) AS observation_date,
  obsdoc.RecordedDtm AS observation_datetime,
  32817 AS observation_type_concept_id,  -- EHR
  CASE
    WHEN obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN CAST(obs.ValueText AS DOUBLE)
    ELSE NULL
  END AS value_as_number,
  CASE
    WHEN obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
    ELSE obs.ValueText
  END AS value_as_string,
  NULL AS value_as_concept_id,
  NULL AS qualifier_concept_id,
  0 AS unit_concept_id,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT('allscripts_scm', ' | ', obs.GUID) AS observation_source_value,
  0 AS observation_source_concept_id,
  obs.UnitOfMeasure AS unit_source_value,
  NULL AS qualifier_source_value,
  obs.ValueText AS value_source_value,
  NULL AS observation_event_id,
  NULL AS obs_event_field_concept_id,
  'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING)) = stp.person_source_value
  AND stp.active_flag = TRUE
LEFT JOIN _exponent.results_store.omop_mapping_scm_observation_row_results_v1 obs_concept
  ON UPPER(obs_concept.entryname) = UPPER(CAST(obs.ObsItemGUID AS STRING))
WHERE obs.GUID IS NOT NULL
  AND obs.StatusType = 1  -- Performed
  AND obs.ValueText IS NOT NULL
  AND obsdoc.RecordedDtm >= '1950-01-01'
  AND NOT (obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$')  -- Non-numeric values only
-- LIMIT 10000

In [0]:
%sql
SELECT * FROM silver_observation LIMIT 100

# Merge to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.observation AS t
USING silver_observation AS s
ON t.observation_source_value = s.observation_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.observation_concept_id <=> s.observation_concept_id)
  OR NOT (t.observation_date <=> s.observation_date)
  OR NOT (t.observation_datetime <=> s.observation_datetime)
  OR NOT (t.observation_type_concept_id <=> s.observation_type_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.value_as_string <=> s.value_as_string)
  OR NOT (t.value_as_concept_id <=> s.value_as_concept_id)
  OR NOT (t.qualifier_concept_id <=> s.qualifier_concept_id)
  OR NOT (t.unit_concept_id <=> s.unit_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.observation_source_concept_id <=> s.observation_source_concept_id)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.qualifier_source_value <=> s.qualifier_source_value)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.observation_event_id <=> s.observation_event_id)
  OR NOT (t.obs_event_field_concept_id <=> s.obs_event_field_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.observation_concept_id         = s.observation_concept_id,
  t.observation_date               = s.observation_date,
  t.observation_datetime           = s.observation_datetime,
  t.observation_type_concept_id    = s.observation_type_concept_id,
  t.value_as_number                = s.value_as_number,
  t.value_as_string                = s.value_as_string,
  t.value_as_concept_id            = s.value_as_concept_id,
  t.qualifier_concept_id           = s.qualifier_concept_id,
  t.unit_concept_id                = s.unit_concept_id,
  t.provider_id                    = s.provider_id,
  t.visit_occurrence_id            = s.visit_occurrence_id,
  t.visit_detail_id                = s.visit_detail_id,
  t.observation_source_concept_id  = s.observation_source_concept_id,
  t.unit_source_value              = s.unit_source_value,
  t.qualifier_source_value         = s.qualifier_source_value,
  t.value_source_value             = s.value_source_value,
  t.observation_event_id           = s.observation_event_id,
  t.obs_event_field_concept_id     = s.obs_event_field_concept_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id,
  source_system
)
VALUES (
  s.person_id,
  s.observation_concept_id,
  s.observation_date,
  s.observation_datetime,
  s.observation_type_concept_id,
  s.value_as_number,
  s.value_as_string,
  s.value_as_concept_id,
  s.qualifier_concept_id,
  s.unit_concept_id,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.observation_source_value,
  s.observation_source_concept_id,
  s.unit_source_value,
  s.qualifier_source_value,
  s.value_source_value,
  s.observation_event_id,
  s.obs_event_field_concept_id,
  s.source_system
);

# Populate Mapping Table

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
    source_system,
    observation_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.observation_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        observation_source_value,
        person_id
    FROM _exponent.omop_silver.observation
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation x
  ON s.observation_source_value = x.observation_source_value;

# Merge to Gold

In [0]:
%sql
-- MERGE INTO _exponent.omop.observation AS gold_obs
MERGE INTO _exponent.omop_scm.observation AS gold_obs
USING (
  SELECT
    source_to_observation.observation_id,
    s.person_id,
    s.observation_concept_id,
    s.observation_date,
    s.observation_datetime,
    s.observation_type_concept_id,
    s.value_as_number,
    s.value_as_string,
    s.value_as_concept_id,
    s.qualifier_concept_id,
    s.unit_concept_id,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.observation_source_value,
    s.observation_source_concept_id,
    s.unit_source_value,
    s.qualifier_source_value,
    s.value_source_value,
    s.observation_event_id,
    s.obs_event_field_concept_id
  FROM _exponent.omop_silver.observation s
  JOIN _exponent.omop_mapping.source_to_observation
    ON source_to_observation.observation_source_value = s.observation_source_value
   AND source_to_observation.active_flag = TRUE
  WHERE 1=1
  AND s.source_system = 'allscripts_scm'
) AS src
ON gold_obs.observation_id = src.observation_id

WHEN MATCHED THEN UPDATE SET
  gold_obs.person_id                     = src.person_id,
  gold_obs.observation_concept_id        = src.observation_concept_id,
  gold_obs.observation_date              = src.observation_date,
  gold_obs.observation_datetime          = src.observation_datetime,
  gold_obs.observation_type_concept_id   = src.observation_type_concept_id,
  gold_obs.value_as_number               = src.value_as_number,
  gold_obs.value_as_string               = src.value_as_string,
  gold_obs.value_as_concept_id           = src.value_as_concept_id,
  gold_obs.qualifier_concept_id          = src.qualifier_concept_id,
  gold_obs.unit_concept_id               = src.unit_concept_id,
  gold_obs.provider_id                   = src.provider_id,
  gold_obs.visit_occurrence_id           = src.visit_occurrence_id,
  gold_obs.visit_detail_id               = src.visit_detail_id,
  gold_obs.observation_source_value      = src.observation_source_value,
  gold_obs.observation_source_concept_id = src.observation_source_concept_id,
  gold_obs.unit_source_value             = src.unit_source_value,
  gold_obs.qualifier_source_value        = src.qualifier_source_value,
  gold_obs.value_source_value            = src.value_source_value,
  gold_obs.observation_event_id          = src.observation_event_id,
  gold_obs.obs_event_field_concept_id    = src.obs_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value,
  src.observation_event_id,
  src.obs_event_field_concept_id
);